<a href="https://colab.research.google.com/github/suhana-2/optiforge-2026-allies/blob/main/ga_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Drift-Robust and Fair Lightweight Classifier via Multi-Objective Evolutionary Hyperparameter Search

Team: allies (OPT-26-9838) | Track: 04 Machine Learning & AI
SDG: SDG 10 (Reduced Inequalities)

A genetic algorithm tunes a RandomForest classifier. Each individual is
[n_estimators, max_depth, min_samples_split].

Parameter configuration
- Seed: 42
- Population size: 20
- Generations: 15
- Mutation rate: 0.2 (per gene)
- Selection: tournament, k = 3
- Crossover: uniform
- Elitism: top 2 copied unchanged

Fitness
0.5 x clean validation accuracy + 0.4 x drift-perturbed accuracy - 0.1 x size penalty (n_estimators x max_depth, normalised)

Attempt log
- Attempt 1: baseline GA, final best fitness 0.8036 with [36, 13, 9]

In [1]:
import random
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [2]:
SEED = 42
POP_SIZE = 20
GENERATIONS = 15
MUTATION_RATE = 0.2

In [3]:
random.seed(SEED)
np.random.seed(SEED)

In [4]:
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=SEED)
X_train, X_val, y_train, y_val= train_test_split(X,y, test_size=0.3, random_state=SEED)
print("Train shape:", X_train.shape)
print("Val Shape:", X_val.shape)

Train shape: (700, 20)
Val Shape: (300, 20)


In [5]:
GENE_RANGES = {
    "n_estimators": (10, 200),
    "max_depth": (2, 20),
    "min_samples_split": (2, 20),
}

def random_individual():
    """Create one random individual within the allowed ranges."""
    return [
        random.randint(*GENE_RANGES["n_estimators"]),
        random.randint(*GENE_RANGES["max_depth"]),
        random.randint(*GENE_RANGES["min_samples_split"]),
    ]

def init_population(size):
    """Create the starting population."""
    return [random_individual() for _ in range(size)]

population = init_population(POP_SIZE)
print("Population size:", len(population))
print("First 3 individuals:", population[:3])

Population size: 20
First 3 individuals: [[173, 5, 2], [199, 10, 9], [67, 6, 5]]


In [6]:
from sklearn.ensemble import RandomForestClassifier
rng = np.random.RandomState(SEED)
X_val_drift = X_val + 0.5 + rng.normal(0, 0.5, X_val.shape)
W_CLEAN, W_DRIFT, W_SIZE = 0.5, 0.4, 0.1
MAX_SIZE = GENE_RANGES["n_estimators"][1] * GENE_RANGES["max_depth"][1]

def fitness(ind):
    """Higher is better. Balances clean accuracy, drift accuracy, and model size."""
    n_est, depth, min_split = ind
    model = RandomForestClassifier(
        n_estimators=n_est, max_depth=depth,
        min_samples_split=min_split, random_state=SEED, n_jobs=-1,
    )
    model.fit(X_train, y_train)
    clean_acc = model.score(X_val, y_val)
    drift_acc = model.score(X_val_drift, y_val)
    size_penalty = (n_est * depth) / MAX_SIZE   # 0 to 1, proxy for model size
    return W_CLEAN * clean_acc + W_DRIFT * drift_acc - W_SIZE * size_penalty

for ind in population[:3]:
    print(ind, "->", round(fitness(ind), 4))

[173, 5, 2] -> 0.7834
[199, 10, 9] -> 0.7626
[67, 6, 5] -> 0.7893


In [7]:
TOURNAMENT_K = 3
ELITE = 2

def tournament_select(pop, scores):
    """Pick K random individuals, return the best one."""
    idx = random.sample(range(len(pop)), TOURNAMENT_K)
    best = max(idx, key=lambda i: scores[i])
    return pop[best][:]

def crossover(p1, p2):
    """Uniform crossover: each gene comes from either parent."""
    return [random.choice(pair) for pair in zip(p1, p2)]

def mutate(ind):
    """With MUTATION_RATE per gene, resample it within its allowed range."""
    keys = list(GENE_RANGES)
    for i, k in enumerate(keys):
        if random.random() < MUTATION_RATE:
            ind[i] = random.randint(*GENE_RANGES[k])
    return ind

def run_ga():
    pop = init_population(POP_SIZE)
    history = []
    for gen in range(GENERATIONS):
        scores = [fitness(ind) for ind in pop]
        ranked = sorted(zip(scores, pop), key=lambda t: t[0], reverse=True)
        best_score, best_ind = ranked[0]
        history.append(best_score)
        print(f"Gen {gen+1:2d} | best fitness {best_score:.4f} | "
              f"mean {np.mean(scores):.4f} | best {best_ind}")

        new_pop = [ind[:] for _, ind in ranked[:ELITE]]
        while len(new_pop) < POP_SIZE:
            child = crossover(tournament_select(pop, scores),
                              tournament_select(pop, scores))
            new_pop.append(mutate(child))
        pop = new_pop
    return best_ind, best_score, history

best_ind, best_score, history = run_ga()
print("\nFINAL BEST:", best_ind, "fitness:", round(best_score, 4))

Gen  1 | best fitness 0.7938 | mean 0.7699 | best [77, 6, 9]
Gen  2 | best fitness 0.7997 | mean 0.7774 | best [64, 6, 14]
Gen  3 | best fitness 0.8007 | mean 0.7917 | best [64, 6, 9]
Gen  4 | best fitness 0.8007 | mean 0.7871 | best [64, 6, 9]
Gen  5 | best fitness 0.8007 | mean 0.7935 | best [64, 6, 9]
Gen  6 | best fitness 0.8007 | mean 0.7868 | best [64, 6, 9]
Gen  7 | best fitness 0.8007 | mean 0.7951 | best [64, 6, 9]
Gen  8 | best fitness 0.8007 | mean 0.7958 | best [64, 6, 9]
Gen  9 | best fitness 0.8007 | mean 0.7958 | best [64, 6, 9]
Gen 10 | best fitness 0.8007 | mean 0.7942 | best [64, 6, 9]
Gen 11 | best fitness 0.8007 | mean 0.7945 | best [64, 6, 9]
Gen 12 | best fitness 0.8007 | mean 0.7929 | best [64, 6, 9]
Gen 13 | best fitness 0.8036 | mean 0.7943 | best [36, 13, 9]
Gen 14 | best fitness 0.8036 | mean 0.7914 | best [36, 13, 9]
Gen 15 | best fitness 0.8036 | mean 0.7914 | best [36, 13, 9]

FINAL BEST: [36, 13, 9] fitness: 0.8036
